In [31]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler


# MODELS
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, recall_score, roc_auc_score, precision_score, accuracy_score, f1_score

In [22]:
df_nondiab = pd.read_csv("nondiabetic_merged_filled.csv")
df_nondiab["diabetes_status"] = 0
df_nondiab.drop(columns=["Respondent_ID"], inplace=True)
df_nondiab.rename(columns={"Sex_num": "Gender"}, inplace=True)
df_nondiab.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 378 entries, 0 to 377
Data columns (total 33 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Gender                 378 non-null    int64  
 1   alcohol_ord            378 non-null    int64  
 2   Age                    378 non-null    int64  
 3   Height_cm              377 non-null    float64
 4   Weight_kg              377 non-null    float64
 5   BMI                    376 non-null    float64
 6   Course                 355 non-null    object 
 7   Year Level             355 non-null    object 
 8   fruit_freq_ord         378 non-null    int64  
 9   veg_freq_ord           378 non-null    int64  
 10  sweets_freq_ord        378 non-null    int64  
 11  fastfood_freq_ord      378 non-null    int64  
 12  processed_freq_ord     378 non-null    int64  
 13  sweetdrink_freq_ord    378 non-null    int64  
 14  fried_food_freq_ord    378 non-null    int64  
 15  lose_w

In [23]:
df_diab = pd.read_csv("diab_merge_final.csv")
df_diab["diabetes_status"] = 1
df_diab.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68 entries, 0 to 67
Data columns (total 29 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    68 non-null     int64  
 1   Gender                 66 non-null     float64
 2   Height_cm              68 non-null     float64
 3   Weight_kg              68 non-null     float64
 4   BMI                    68 non-null     float64
 5   fruit_freq_ord         67 non-null     float64
 6   veg_freq_ord           67 non-null     float64
 7   sweets_freq_ord        66 non-null     float64
 8   fastfood_freq_ord      67 non-null     float64
 9   processed_freq_ord     66 non-null     float64
 10  sweetdrink_freq_ord    67 non-null     float64
 11  fried_food_freq_ord    65 non-null     float64
 12  lose_weight_ord        66 non-null     float64
 13  exercise_yes_ord       66 non-null     float64
 14  exercise_freq_ord      64 non-null     float64
 15  exercise

In [24]:
merged_df = pd.concat([df_nondiab, df_diab], ignore_index=True)

merged_df.drop(columns=["BMI", "child_diab_ord", "Course", "Year Level", "Height_cm", "Weight_kg", "Age", "Gender", "alcohol_ord.1", "concern_level_ord"], inplace=True)
merged_df.info()
merged_df["diabetes_status"].value_counts()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 446 entries, 0 to 445
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   alcohol_ord            443 non-null    float64
 1   fruit_freq_ord         445 non-null    float64
 2   veg_freq_ord           445 non-null    float64
 3   sweets_freq_ord        444 non-null    float64
 4   fastfood_freq_ord      445 non-null    float64
 5   processed_freq_ord     444 non-null    float64
 6   sweetdrink_freq_ord    445 non-null    float64
 7   fried_food_freq_ord    443 non-null    float64
 8   lose_weight_ord        443 non-null    float64
 9   exercise_yes_ord       444 non-null    float64
 10  exercise_freq_ord      442 non-null    float64
 11  exercise_duration_ord  445 non-null    float64
 12  sedentary_hours_ord    440 non-null    float64
 13  activity_level_ord     443 non-null    float64
 14  transpo_ord            378 non-null    float64
 15  sleep_

diabetes_status
0    378
1     68
Name: count, dtype: int64

In [35]:


merged_df.fillna(merged_df.median(numeric_only=True), inplace=True)
merged_df.info()
merged_df["diabetes_status"].value_counts()
merged_df.columns


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 446 entries, 0 to 445
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   alcohol_ord            446 non-null    float64
 1   fruit_freq_ord         446 non-null    float64
 2   veg_freq_ord           446 non-null    float64
 3   sweets_freq_ord        446 non-null    float64
 4   fastfood_freq_ord      446 non-null    float64
 5   processed_freq_ord     446 non-null    float64
 6   sweetdrink_freq_ord    446 non-null    float64
 7   fried_food_freq_ord    446 non-null    float64
 8   lose_weight_ord        446 non-null    float64
 9   exercise_yes_ord       446 non-null    float64
 10  exercise_freq_ord      446 non-null    float64
 11  exercise_duration_ord  446 non-null    float64
 12  sedentary_hours_ord    446 non-null    float64
 13  activity_level_ord     446 non-null    float64
 14  transpo_ord            446 non-null    float64
 15  sleep_

Index(['alcohol_ord', 'fruit_freq_ord', 'veg_freq_ord', 'sweets_freq_ord',
       'fastfood_freq_ord', 'processed_freq_ord', 'sweetdrink_freq_ord',
       'fried_food_freq_ord', 'lose_weight_ord', 'exercise_yes_ord',
       'exercise_freq_ord', 'exercise_duration_ord', 'sedentary_hours_ord',
       'activity_level_ord', 'transpo_ord', 'sleep_ord', 'smoking_ord',
       'father_diab_ord', 'mother_diab_ord', 'sister_diab_ord',
       'brother_diab_ord', 'extended_diab_ord', 'diabetes_status'],
      dtype='object')

In [ ]:
X = merged_df.drop(columns=["diabetes_status"])
y = merged_df["diabetes_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_train.columns
y.value_counts()



TypeError: 'Index' object is not callable

In [27]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(probability=True, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

In [28]:
numeric_features = [ 'sweets_freq_ord', 'fruit_freq_ord', 'veg_freq_ord', 'fastfood_freq_ord',
    'processed_freq_ord', 'sweetdrink_freq_ord', 'fried_food_freq_ord',
    'lose_weight_ord', 'exercise_yes_ord', 'exercise_freq_ord', 'exercise_duration_ord'
,'sedentary_hours_ord',"activity_level_ord", "transpo_ord", "sleep_ord", "alcohol_ord", "father_diab_ord", "mother_diab_ord", "sister_diab_ord", "brother_diab_ord","extended_diab_ord"]

over = SMOTE()
under = RandomUnderSampler()

preprocessor = ColumnTransformer(
    transformers=[('num', MinMaxScaler(), numeric_features)],
    remainder='passthrough'  # keep any non-numeric features if present
)

print(numeric_features)

['sweets_freq_ord', 'fruit_freq_ord', 'veg_freq_ord', 'fastfood_freq_ord', 'processed_freq_ord', 'sweetdrink_freq_ord', 'fried_food_freq_ord', 'lose_weight_ord', 'exercise_yes_ord', 'exercise_freq_ord', 'exercise_duration_ord', 'sedentary_hours_ord', 'activity_level_ord', 'transpo_ord', 'sleep_ord', 'alcohol_ord', 'father_diab_ord', 'mother_diab_ord', 'sister_diab_ord', 'brother_diab_ord', 'extended_diab_ord']


In [29]:
def train_and_evaluate_model_SMOTE(models, X_train, y_train, X_valid, y_valid):
    """
    Enhanced version that returns model performance for comparison
    """
    results = {}

    for name, model in models.items():
        print(f"\n🔹 Training {name}...")

        if name == "Random Forest" or name == "XGBoost":
          pipe = ImbPipeline([
              ('over', over),
              ('model', model)
          ])
        else:
          pipe = ImbPipeline([
              ('preprocess', preprocessor),
              ('over', over),
              ('model', model)
          ])

        pipe.fit(X_train, y_train)

        y_pred = pipe.predict(X_valid)
        y_proba = pipe.predict_proba(X_valid)[:, 1]

        # Calculate multiple metrics
        roc_auc = roc_auc_score(y_valid, y_proba)
        accuracy = accuracy_score(y_valid, y_pred)
        precision = precision_score(y_valid, y_pred, zero_division=0)
        recall = recall_score(y_valid, y_pred, zero_division=0)
        f1 = f1_score(y_valid, y_pred, zero_division=0)

        # Store results
        results[name] = {
            'model': pipe,
            'roc_auc': roc_auc,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'predictions': y_pred,
            'probabilities': y_proba
        }

        print(f"ROC-AUC: {roc_auc:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print(classification_report(y_valid, y_pred, digits=2))

        cm = confusion_matrix(y_valid, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot(cmap='Blues')
        plt.title(f"{name} - Confusion Matrix")
        plt.show()

    return results

In [ ]:
results = train_and_evaluate_model_SMOTE(models, X_train, y_train, X_test, y_test)

# Find best model
best_model_name = max(results.keys(), key=lambda x: results[x]['roc_auc'])
best_model = results[best_model_name]['model']
print(f"\n🏆 BEST MODEL: {best_model_name} with ROC-AUC: {results[best_model_name]['roc_auc']:.4f}")

In [36]:
best_pipe = ImbPipeline([
    ('preprocess', preprocessor),
    ('over', over),
    ('model', SVC(probability=True, random_state=42))
])
import joblib

best_pipe.fit(X_train, y_train)
joblib.dump(best_pipe, "best_pipe_lifestyle.pkl")
print("✅ Saved best_pipe.pkl successfully!")

✅ Saved best_pipe.pkl successfully!
